In [ ]:
# fMRI Viewer

import math
import os
import csv
from datetime import datetime
from functools import lru_cache

import nibabel as nib
import numpy as np
import plotly.express as px
from dash import ALL, Dash, Input, Output, State, dcc, html
from dash.exceptions import PreventUpdate


# =========================
# 1) CONFIG
# =========================
# BASE_DIR = "/Users/cassidyschuman/Downloads/Classes/Senior Spring/Clinical Preceptorship/example_output"
BASE_DIR = "/Users/forrestlin/Downloads/example_output"

AXIS_LABELS = {
    0: "Sagittal",
    1: "Coronal",
    2: "Axial",
}

NETWORK_GRADIENTS = [
    (np.array([1.00, 0.00, 0.00]), np.array([0.00, 0.00, 1.00])),  # red → blue
    (np.array([0.00, 1.00, 0.00]), np.array([1.00, 0.00, 1.00])),  # green → magenta
    (np.array([1.00, 0.50, 0.00]), np.array([0.00, 0.00, 1.00])),  # orange → blue
    (np.array([1.00, 1.00, 0.00]), np.array([0.50, 0.00, 1.00])),  # yellow → violet
    (np.array([0.00, 1.00, 1.00]), np.array([1.00, 0.00, 0.00])),  # cyan → red
    (np.array([1.00, 0.00, 0.50]), np.array([0.00, 1.00, 0.50])),  # hot-pink → mint
    (np.array([0.00, 0.00, 1.00]), np.array([1.00, 0.80, 0.00])),  # blue → gold
    (np.array([1.00, 0.00, 1.00]), np.array([0.00, 1.00, 0.00])),  # magenta → green
    (np.array([0.00, 0.80, 1.00]), np.array([1.00, 0.30, 0.00])),  # sky-blue → burnt-orange
    (np.array([0.60, 0.00, 1.00]), np.array([0.00, 1.00, 0.40])),  # purple → lime
    (np.array([1.00, 0.20, 0.00]), np.array([0.00, 0.80, 1.00])),  # scarlet → azure
    (np.array([0.00, 1.00, 0.60]), np.array([1.00, 0.00, 0.40])),  # spring-green → rose-red
]

GRADIENT_PRESETS = {
    "red-blue": (np.array([1, 0, 0]), np.array([0, 0, 1])),
    "green-magenta": (np.array([0, 1, 0]), np.array([1, 0, 1])),
    "orange-blue": (np.array([1, 0.5, 0]), np.array([0, 0, 1])),
    "yellow-violet": (np.array([1, 1, 0]), np.array([0.5, 0, 1])),
    "cyan-red": (np.array([0, 1, 1]), np.array([1, 0, 0])),
    "blue-gold": (np.array([0, 0, 1]), np.array([1, 0.8, 0])),
}

SUFFIX_DISPLAY_NAMES = {
    "desc-brain_mask_t1":   "Brain Mask",
    "reho_t1":              "ReHo",
    "aud_net_t1":           "Auditory Network",
    "ec_net_t1":            "EC Network",
    "lfp_net_t1":           "Left Frontoparietal Network",
    "rfp_net_t1":           "Right Frontoparietal Network",
    "vl_net_t1":            "Visuolateral Network",
    "vm_net_t1":            "Visuomedial Network",
    "vo_net_t1":            "Visuooccipital Network",
    "dmn_net_t1":           "Default Mode Network",
    "seedcorr_t1":          "Seed Correlation (T1w)",
    "sm_net_t1":            "Sensorimotor Network",
}

SUFFIX_GRADIENT_INDEX = {
    "aud_net_t1":           0,   # red → blue
    "desc-brain_mask_t1":   1,   # green → magenta
    "dmn_net_t1":           2,   # orange → blue
    "ec_net_t1":            3,   # yellow → violet
    "lfp_net_t1":           4,   # cyan → red
    "reho_t1":              5,   # hot-pink → mint
    "rfp_net_t1":           6,   # blue → gold
    "seedcorr_t1":          7,   # magenta → green
    "vl_net_t1":            8,   # sky-blue → burnt-orange
    "vm_net_t1":            9,   # purple → lime
    "vo_net_t1":            10,  # scarlet → azure
    "sm_net_t1":            11,  # spring-green → rose-red
}

CARD_STYLE = {
    "backgroundColor": "#f4f4f4",
    "borderRadius": "16px",
    "padding": "20px",
    "boxShadow": "0 1px 6px rgba(0,0,0,0.08)",
}

SECTION_TITLE_STYLE = {
    "fontSize": "18px",
    "fontWeight": "700",
    "marginBottom": "14px",
}

LABEL_STYLE = {
    "fontSize": "14px",
    "fontWeight": "600",
    "marginBottom": "8px",
    "display": "block",
}

DEFAULT_OPACITY = 0.70
DEFAULT_THRESHOLD = 0.50


# =========================
# 2) DATA DISCOVERY
# =========================
def list_patients():
    if not os.path.isdir(BASE_DIR):
        return []
    return sorted(
        d for d in os.listdir(BASE_DIR)
        if os.path.isdir(os.path.join(BASE_DIR, d))
    )


PATIENTS = list_patients()

_fname_to_grad: dict[str, int] = {}


def patient_display_dir(patient_id):
    return os.path.join(BASE_DIR, patient_id, "xprx", "display")


def patient_root_dir(patient_id):
    return os.path.join(BASE_DIR, patient_id)


def get_display_niftis(patient_id):
    display_path = patient_display_dir(patient_id)
    if not os.path.isdir(display_path):
        return {}

    nii_files = [
        f for f in os.listdir(display_path)
        if f.endswith(".nii") or f.endswith(".nii.gz")
    ]

    file_dict = {}
    prefix = f"sub-{patient_id}_ses-1_task-rest_acq-multiecho_"

    for f in sorted(nii_files):
        name = f.replace(".nii.gz", "").replace(".nii", "")
        suffix = name.replace(prefix, "") if name.startswith(prefix) else name
        display_label = SUFFIX_DISPLAY_NAMES.get(suffix, suffix)
        gradient_idx = SUFFIX_GRADIENT_INDEX.get(suffix, 0)
        file_dict[display_label] = (f, gradient_idx)
        _fname_to_grad[f] = gradient_idx

    return file_dict


for _p in PATIENTS:
    get_display_niftis(_p)


def find_t1_file(patient_id):
    root = patient_root_dir(patient_id)
    if not os.path.isdir(root):
        return None

    preferred_terms = ["desc-preproc_T1w", "T1w", "T1", "t1w", "t1"]
    candidates = []

    for dirpath, _, filenames in os.walk(root):
        for fname in filenames:
            if not (fname.endswith(".nii") or fname.endswith(".nii.gz")):
                continue
            full_path = os.path.join(dirpath, fname)
            score = 0
            for i, term in enumerate(preferred_terms):
                if term in fname:
                    score = max(score, len(preferred_terms) - i)
            if score > 0:
                candidates.append((score, full_path))

    if not candidates:
        return None
    candidates.sort(key=lambda x: (-x[0], x[1]))
    return candidates[0][1]


# =========================
# 3) VOLUME HELPERS
# =========================
@lru_cache(maxsize=256)
def load_nifti(path):
    img = nib.load(path)
    vol = img.get_fdata().astype(np.float32)
    if vol.ndim == 4:
        vol = vol[..., 0]
    return np.nan_to_num(vol)


@lru_cache(maxsize=256)
def load_patient_overlay(patient_id, filename):
    path = os.path.join(patient_display_dir(patient_id), filename)
    return load_nifti(path)


@lru_cache(maxsize=64)
def load_patient_t1(patient_id):
    t1_path = find_t1_file(patient_id)
    if t1_path is None:
        return None
    return load_nifti(t1_path)


def normalize_to_unit(arr):
    arr = np.asarray(arr, dtype=np.float32)
    vmin, vmax = float(np.min(arr)), float(np.max(arr))
    if vmax <= vmin:
        return np.zeros_like(arr)
    return (arr - vmin) / (vmax - vmin)


def clip_percentile(sl, p_low=1, p_high=99):
    lo, hi = np.percentile(sl, (p_low, p_high))
    if hi <= lo:
        return np.zeros_like(sl, dtype=np.float32)
    return np.clip(sl, lo, hi).astype(np.float32)


def get_slice(vol3d, axis, idx):
    if axis == 0:
        sl = vol3d[idx, :, :]
    elif axis == 1:
        sl = vol3d[:, idx, :]
    else:
        sl = vol3d[:, :, idx]
    return np.rot90(sl)


def make_mosaic(vol3d, axis=2, start=0, step=1, n_slices=36, pad=2):
    max_idx = vol3d.shape[axis] - 1
    start = int(np.clip(start, 0, max_idx))
    indices = list(range(start, max_idx + 1, max(1, step)))[:max(1, n_slices)]
    if not indices:
        indices = [start]

    slices = [clip_percentile(get_slice(vol3d, axis, i), 1, 99) for i in indices]
    cols = int(math.ceil(math.sqrt(len(slices))))
    rows = int(math.ceil(len(slices) / cols))
    h, w = slices[0].shape
    mosaic = np.zeros((rows * h + (rows - 1) * pad, cols * w + (cols - 1) * pad), dtype=np.float32)
    for k, sl in enumerate(slices):
        r, c = k // cols, k % cols
        mosaic[r * (h + pad):r * (h + pad) + h, c * (w + pad):c * (w + pad) + w] = sl
    return mosaic, indices


def gradient_color(t, low_color, high_color):
    t = np.asarray(t, dtype=np.float32)[..., np.newaxis]
    return (1.0 - t) * low_color + t * high_color


def blend_single_slice(base_vol, overlay_vols, axis, idx, per_overlay_settings, grad_indices, gradient_settings, selected_files):
    base_slice = normalize_to_unit(clip_percentile(get_slice(base_vol, axis, idx), 1, 99))
    rgb = np.stack([base_slice, base_slice, base_slice], axis=-1)

    for i, vol in enumerate(overlay_vols):
        settings = per_overlay_settings[i]
        threshold = float(settings.get("threshold", DEFAULT_THRESHOLD))
        alpha = float(np.clip(settings.get("opacity", DEFAULT_OPACITY), 0, 1))

        overlay_slice = normalize_to_unit(get_slice(vol, axis, idx))
        mask = overlay_slice >= threshold
        if not np.any(mask):
            continue

        fname = selected_files[i] if i < len(selected_files) else None
        custom_grad = (gradient_settings or {}).get(fname, {}).get("gradient")

        entry = (gradient_settings or {}).get(fname, {})

        if "low" in entry and "high" in entry:
            low_c = np.array(entry["low"])
            high_c = np.array(entry["high"])
        elif entry.get("gradient") in GRADIENT_PRESETS:
            low_c, high_c = GRADIENT_PRESETS[entry["gradient"]]
        else:
            low_c, high_c = NETWORK_GRADIENTS[grad_indices[i]]
        t_vals = np.zeros_like(overlay_slice)
        above = overlay_slice[mask]
        if above.max() > threshold:
            t_vals[mask] = (above - threshold) / (1.0 - threshold + 1e-8)

        grad_colors = gradient_color(t_vals, low_c, high_c)
        rgb[mask] = (1.0 - alpha) * rgb[mask] + alpha * grad_colors[mask]

    return np.clip(rgb, 0, 1)


def _css_gradient(low_color, high_color):
    lo = tuple((255 * np.clip(low_color, 0, 1)).astype(int))
    hi = tuple((255 * np.clip(high_color, 0, 1)).astype(int))
    return f"linear-gradient(to right, rgb{lo}, rgb{hi})"


def empty_figure(message="No image available"):
    fig = px.imshow(np.zeros((16, 16)))
    fig.update_layout(
        template="plotly_white",
        xaxis={"visible": False},
        yaxis={"visible": False},
        coloraxis_showscale=False,
        annotations=[dict(text=message, x=0.5, y=0.5, xref="paper", yref="paper",
                          showarrow=False, font=dict(size=18, color="#666"))],
        margin=dict(l=10, r=10, t=10, b=10),
    )
    return fig

def hex_to_rgb_array(hex_color):
    hex_color = hex_color.lstrip("#")
    return np.array([int(hex_color[i:i+2], 16) for i in (0, 2, 4)]) / 255.0



# =========================
# 4) APP
# =========================
app = Dash(__name__)
app.title = "fMRI Visualization"

default_patient = PATIENTS[0] if PATIENTS else None

app.layout = html.Div(
    style={
        "backgroundColor": "#ececec",
        "minHeight": "100vh",
        "padding": "16px",
        "fontFamily": "Inter, system-ui, -apple-system, Segoe UI, Roboto, sans-serif",
    },
    children=[
        # Per-overlay settings store: {fname: {"opacity": float, "threshold": float}}
        dcc.Store(id="overlay_settings", data={}),
        dcc.Store(id="network_ratings", data={}),
        dcc.Store(id="gradient_settings", data={}),

        # Header
        html.Div(
            style={**CARD_STYLE, "marginBottom": "16px", "padding": "22px 28px",
                   "display": "flex", "justifyContent": "center", "alignItems": "center"},
            children=[
                html.H1("fMRI Visualization",
                        style={"margin": 0, "fontSize": "42px", "fontWeight": "800", "letterSpacing": "0.5px"})
            ],
        ),

        # Main grid
        html.Div(
            style={
                "display": "grid",
                "gridTemplateColumns": "300px minmax(700px, 1fr) 340px",
                "gap": "16px",
                "alignItems": "start",
            },
            children=[
                # ── LEFT COLUMN ──────────────────────────────────────────
                html.Div(
                    style={"display": "flex", "flexDirection": "column", "gap": "16px"},
                    children=[
                        html.Div(
                            style=CARD_STYLE,
                            children=[
                                html.Div("Patient Selection", style={**SECTION_TITLE_STYLE, "fontSize": "20px"}),
                                html.Label("Patient", style=LABEL_STYLE),
                                dcc.Dropdown(
                                    id="patient_dropdown",
                                    options=[{"label": p, "value": p} for p in PATIENTS],
                                    value=default_patient,
                                    clearable=False,
                                ),
                                html.Div(style={"height": "16px"}),
                                html.Label("Active Overlays", style=LABEL_STYLE),
                                dcc.Dropdown(
                                    id="nifti_dropdown",
                                    multi=True,
                                    placeholder="Select overlays to display",
                                ),
                            ],
                        ),
                        html.Div(
                            style=CARD_STYLE,
                            children=[
                                html.Div("Viewing Controls", style={**SECTION_TITLE_STYLE, "fontSize": "20px"}),
                                html.Label("View Mode", style=LABEL_STYLE),
                                dcc.RadioItems(
                                    id="mode",
                                    options=[
                                        {"label": "Single slice", "value": "single"},
                                        {"label": "Mosaic", "value": "mosaic"},
                                    ],
                                    value="single",
                                    labelStyle={"display": "block", "marginBottom": "8px"},
                                    inputStyle={"marginRight": "8px"},
                                ),
                                html.Div(style={"height": "18px"}),
                                html.Label("Plane", style=LABEL_STYLE),
                                dcc.RadioItems(
                                    id="axis",
                                    options=[
                                        {"label": "Sagittal", "value": 0},
                                        {"label": "Coronal", "value": 1},
                                        {"label": "Axial", "value": 2},
                                    ],
                                    value=2,
                                    inline=True,
                                    labelStyle={"marginRight": "16px"},
                                    inputStyle={"marginRight": "6px"},
                                ),
                            ],
                        ),
                    ],
                ),

                # ── CENTER COLUMN ─────────────────────────────────────────
                html.Div(
                    style={"display": "flex", "flexDirection": "column", "gap": "16px"},
                    children=[
                        html.Div(
                            style=CARD_STYLE,
                            children=[
                                html.Div(
                                    style={"display": "flex", "justifyContent": "space-between",
                                           "alignItems": "center", "marginBottom": "10px"},
                                    children=[
                                        html.Div("Image Viewer", style={**SECTION_TITLE_STYLE, "marginBottom": 0}),
                                        html.Div(id="note", style={"fontSize": "13px", "color": "#666", "textAlign": "right"}),
                                    ],
                                ),
                                dcc.Graph(
                                    id="view",
                                    style={"height": "760px"},
                                    config={"displayModeBar": False},
                                    figure=empty_figure("Select a patient to begin"),
                                ),
                                html.Div(style={"height": "6px"}),
                                html.Label("Slice", style=LABEL_STYLE),
                                dcc.Slider(id="idx", min=0, max=100, step=1, value=50, updatemode="drag"),
                                html.Div(style={"height": "20px"}),
                                html.Label("Mosaic slices", style=LABEL_STYLE),
                                dcc.Slider(
                                    id="mosaic_n", min=4, max=100, step=4, value=36,
                                    marks={4: "4", 16: "16", 36: "36", 64: "64", 100: "100"},
                                    updatemode="drag",
                                ),
                            ],
                        ),
                    ],
                ),

                # ── RIGHT COLUMN ──────────────────────────────────────────
                html.Div(
                    style={"display": "flex", "flexDirection": "column", "gap": "16px"},
                    children=[
                        html.Div(
                            style=CARD_STYLE,
                            children=[
                                html.Div("Network Overlay", style={**SECTION_TITLE_STYLE, "fontSize": "20px"}),
                                html.Div(id="network_legend"),
                            ],
                        ),
                        html.Div(
                            style=CARD_STYLE,
                            children=[
                                html.Div("Overlay Controls", style={**SECTION_TITLE_STYLE, "fontSize": "20px"}),
                                html.Label("Select Active Overlay", style=LABEL_STYLE),
                                dcc.Dropdown(
                                    id="active_overlay_dropdown",
                                    placeholder="Pick an overlay to adjust",
                                    clearable=True,
                                ),
                                html.Div(style={"height": "18px"}),
                                html.Div(id="active_overlay_controls"),
                            ],
                        ),
                    ],
                ),
            ],
        ),
    ],
)


# =========================
# 5) CALLBACKS
# =========================

@app.callback(
    Output("nifti_dropdown", "options"),
    Output("nifti_dropdown", "value"),
    Input("patient_dropdown", "value"),
)
def update_nifti_dropdown(patient_id):
    if not patient_id:
        return [], []
    file_dict = get_display_niftis(patient_id)
    if not file_dict:
        return [], []
    options = [{"label": label, "value": fname} for label, (fname, _) in sorted(file_dict.items())]
    return options, []


@app.callback(
    Output("active_overlay_dropdown", "options"),
    Output("active_overlay_dropdown", "value"),
    Input("nifti_dropdown", "value"),
    State("nifti_dropdown", "options"),
    State("active_overlay_dropdown", "value"),
)
def sync_active_overlay_options(selected_files, nifti_options, current_active):
    selected_files = selected_files or []
    label_lookup = {opt["value"]: opt["label"] for opt in (nifti_options or [])}
    options = [{"label": label_lookup.get(f, f), "value": f} for f in selected_files]
    new_active = current_active if current_active in selected_files else (selected_files[0] if selected_files else None)
    return options, new_active


@app.callback(
    Output("active_overlay_controls", "children"),
    Input("active_overlay_dropdown", "value"),
    State("nifti_dropdown", "options"),
    State("overlay_settings", "data"),
    State("network_ratings", "data"),
    State("gradient_settings","data"),
)
def render_active_controls(active_fname, nifti_options, settings, ratings, gradient_settings):
    if not active_fname:
        return html.Div("No overlay selected.", style={"color": "#666", "fontSize": "14px"})

    label_lookup = {opt["value"]: opt["label"] for opt in (nifti_options or [])}
    label = label_lookup.get(active_fname, active_fname)
    g_idx = _fname_to_grad.get(active_fname, 0)
    custom = (gradient_settings or {}).get(active_fname, {})
    def rgb_to_hex(rgb):
        return "#" + "".join(f"{int(255*x):02x}" for x in rgb)

    low_hex = "#ff0000"
    high_hex = "#0000ff"

    if "low" in custom and "high" in custom:
        low_hex = rgb_to_hex(custom["low"])
        high_hex = rgb_to_hex(custom["high"])
    if "low" in custom and "high" in custom:
        low_c = np.array(custom["low"])
        high_c = np.array(custom["high"])
    elif custom.get("gradient") in GRADIENT_PRESETS:
        low_c, high_c = GRADIENT_PRESETS[custom["gradient"]]
    else:
        low_c, high_c = NETWORK_GRADIENTS[g_idx]
    grad_css = _css_gradient(low_c, high_c)

    saved = (settings or {}).get(active_fname, {})
    opacity_val = saved.get("opacity", DEFAULT_OPACITY)
    threshold_val = saved.get("threshold", DEFAULT_THRESHOLD)
    ratings = ratings or {}
    current_rating = ratings.get(active_fname)
    gradient_settings = gradient_settings or {}
    current_grad = gradient_settings.get(active_fname, {}).get("gradient", None)
    return html.Div([
        html.Div(
            style={"display": "flex", "alignItems": "center", "gap": "10px", "marginBottom": "16px"},
            children=[
                html.Div(style={"width": "48px", "height": "14px", "borderRadius": "3px",
                                "background": grad_css, "flexShrink": 0}),
                html.Div(label, style={"fontWeight": "700", "fontSize": "15px"}),
            ],
        ),
        html.Label("Opacity", style=LABEL_STYLE),
        dcc.Slider(
            id="active_opacity_slider",
            min=0, max=1, step=0.05,
            value=opacity_val,
            marks={0: "0", 0.5: "0.5", 1: "1"},
            updatemode="drag",
            tooltip={"placement": "bottom", "always_visible": True},
        ),
        html.Div(style={"height": "18px"}),
        html.Label("Threshold", style=LABEL_STYLE),
        dcc.Slider(
            id="active_threshold_slider",
            min=0, max=1, step=0.01,
            value=threshold_val,
            marks={0: "0", 0.5: "0.5", 1: "1"},
            updatemode="drag",
            tooltip={"placement": "bottom", "always_visible": True},
        ),
        html.Div(
            id="active_threshold_text",
            style={"marginTop": "10px", "fontSize": "13px", "color": "#555"},
            children=f"{threshold_val:.2f} ({int(round(threshold_val * 100))}% of max)",
        ),
        html.Div(style={"height": "18px"}),

        html.Label("Network Rating", style=LABEL_STYLE),
        dcc.Dropdown(
            id="network_rating_dropdown",
            options=[
                {"label": "0", "value": 0},
                {"label": "1", "value": 1},
                {"label": "2", "value": 2},
                {"label": "3", "value": 3},
            ],
            placeholder="Select rating",
            value=current_rating,
            clearable=True,
        ),

        html.Div(style={"height": "20px"}),

        html.Label("Custom Gradient", style=LABEL_STYLE),

        html.Div([
            dcc.Input(
                id="low_color_picker",
                type="color",
                value=low_hex,
                style={"marginRight": "10px"}
            ),
            dcc.Input(
                id="high_color_picker",
                type="color",
                value=high_hex
            ),
        ]),
        html.Button(
            "Save Ratings to CSV",
            id="save_ratings_button",
            n_clicks=0,
            style={
                "backgroundColor": "#4CAF50",
                "color": "white",
                "border": "none",
                "padding": "10px 16px",
                "borderRadius": "8px",
                "cursor": "pointer",
                "fontWeight": "600",
            }
        ),
    ])

@app.callback(
    Output("gradient_settings", "data"),
    Input("low_color_picker", "value"),
    Input("high_color_picker", "value"),
    State("active_overlay_dropdown", "value"),
    State("gradient_settings", "data"),
    prevent_initial_call=True,
)

def save_gradient_setting(low_hex, high_hex, active_fname, current):
    if not active_fname:
        raise PreventUpdate

    data = dict(current or {})
    entry = data.get(active_fname, {})

    if low_hex and high_hex:
        entry["low"] = hex_to_rgb_array(low_hex).tolist()
        entry["high"] = hex_to_rgb_array(high_hex).tolist()

    data[active_fname] = entry

    return data

@app.callback(
    Output("active_threshold_text", "children"),
    Input("active_threshold_slider", "value"),
    prevent_initial_call=True,
)
def update_threshold_text(threshold):
    if threshold is None:
        raise PreventUpdate
    threshold = float(threshold)
    return f"{threshold:.2f} ({int(round(threshold * 100))}% of max)"

@app.callback(
    Output("network_ratings", "data"),
    Input("network_rating_dropdown", "value"),
    State("active_overlay_dropdown", "value"),
    State("network_ratings", "data"),
    prevent_initial_call=True,
)
def save_network_rating(rating, active_fname, current_ratings):
    if active_fname is None:
        raise PreventUpdate

    ratings = dict(current_ratings or {})
    ratings[active_fname] = rating
    return ratings

@app.callback(
    Output("save_ratings_button", "children"),
    Input("save_ratings_button", "n_clicks"),
    State("network_ratings", "data"),
    State("patient_dropdown", "value"),
    State("nifti_dropdown", "value"),
    prevent_initial_call=True,
)
def save_ratings_to_csv(n_clicks, ratings, patient_id, selected_files):
    if not ratings or not patient_id:
        raise PreventUpdate

    selected_files = set(selected_files or [])

    filename = f"network_ratings_{patient_id}_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"
    filepath = os.path.join(BASE_DIR, filename)

    with open(filepath, "w", newline="") as csvfile:
        writer = csv.writer(csvfile)
        writer.writerow(["patient_id", "network_map", "rating"])

        for fname, rating in ratings.items():
            # ❗ only include if still selected
            if fname not in selected_files:
                continue

            label = fname  # or your label_lookup if you want
            writer.writerow([patient_id, label, rating])

    return "Saved!"

def update_threshold_text(threshold):
    if threshold is None:
        raise PreventUpdate
    threshold = float(threshold)
    return f"{threshold:.2f} ({int(round(threshold * 100))}% of max)"


# Save to store immediately whenever either slider changes
@app.callback(
    Output("overlay_settings", "data"),
    Input("active_opacity_slider", "value"),
    Input("active_threshold_slider", "value"),
    State("active_overlay_dropdown", "value"),
    State("overlay_settings", "data"),
    prevent_initial_call=True,
)
def save_overlay_settings(opacity, threshold, active_fname, current_settings):
    if not active_fname:
        raise PreventUpdate
    settings = dict(current_settings or {})
    settings[active_fname] = {
        "opacity": float(opacity) if opacity is not None else DEFAULT_OPACITY,
        "threshold": float(threshold) if threshold is not None else DEFAULT_THRESHOLD,
    }
    return settings


@app.callback(
    Output("network_legend", "children"),
    Input("nifti_dropdown", "value"),
    State("nifti_dropdown", "options"),
    State("gradient_settings", "data"),
)

def update_legend(selected_files, options):
    selected_files = selected_files or []
    if not selected_files:
        return html.Div("No overlays selected.", style={"color": "#666"})

    label_lookup = {opt["value"]: opt["label"] for opt in (options or [])}
    items = [
        html.Div(
            style={"display": "flex", "alignItems": "center", "gap": "10px", "marginBottom": "10px"},
            children=[
                html.Div(style={"width": "48px", "height": "14px", "borderRadius": "3px",
                                "background": "linear-gradient(to right, #888, #fff)"}),
                html.Div("T1 anatomical base", style={"fontSize": "14px", "fontWeight": "600"}),
            ],
        )
    ]
    for fname in selected_files:
        label = label_lookup.get(fname, fname)
        g_idx = _fname_to_grad.get(fname, 0)
        custom_grad = gradient_settings.get(fname, {}).get("gradient") if gradient_settings else None

        entry = (gradient_settings or {}).get(fname, {})

        if "low" in entry and "high" in entry:
            low_c = np.array(entry["low"])
            high_c = np.array(entry["high"])
        elif entry.get("gradient") in GRADIENT_PRESETS:
            low_c, high_c = GRADIENT_PRESETS[entry["gradient"]]
        else:
            low_c, high_c = NETWORK_GRADIENTS[g_idx]
        items.append(
            html.Div(
                style={"display": "flex", "alignItems": "center", "gap": "10px", "marginBottom": "10px"},
                children=[
                    html.Div(style={"width": "48px", "height": "14px", "borderRadius": "3px",
                                    "background": _css_gradient(low_c, high_c)}),
                    html.Div(label, style={"fontSize": "14px", "fontWeight": "600"}),
                ],
            )
        )
    return items


@app.callback(
    Output("idx", "max"),
    Output("idx", "value"),
    Output("note", "children"),
    Input("patient_dropdown", "value"),
    Input("nifti_dropdown", "value"),
    Input("axis", "value"),
    Input("mode", "value"),
    Input("mosaic_n", "value"),
    prevent_initial_call=False,
)
def sync_axis(patient_id, selected_files, axis, mode, mosaic_n):
    if not patient_id or axis is None:
        raise PreventUpdate

    base_vol = load_patient_t1(patient_id)
    if base_vol is None:
        if not selected_files:
            raise PreventUpdate
        base_vol = load_patient_overlay(patient_id, selected_files[0])

    axis = int(axis)
    max_idx = base_vol.shape[axis] - 1
    note = f"T1 base: {base_vol.shape} | {AXIS_LABELS[axis]} | mode: {mode}"
    if mode == "mosaic":
        note += f" | mosaic_n: {mosaic_n}"
    return max_idx, max_idx // 2, note


@app.callback(
    Output("view", "figure"),
    Input("patient_dropdown", "value"),
    Input("nifti_dropdown", "value"),
    Input("overlay_settings", "data"),
    Input("mode", "value"),
    Input("axis", "value"),
    Input("idx", "value"),
    Input("mosaic_n", "value"),
    Input("gradient_settings", "data"),
)
def update_view(patient_id, selected_files, settings, mode, axis, idx, mosaic_n, gradient_settings):
    if not patient_id or mode is None or axis is None or idx is None or mosaic_n is None:
        raise PreventUpdate

    axis = int(axis)
    idx = int(idx)
    mosaic_n = int(mosaic_n)
    settings = settings or {}

    base_vol = load_patient_t1(patient_id)
    if base_vol is None:
        if not selected_files:
            return empty_figure("No T1 file found and no overlay selected")
        base_vol = load_patient_overlay(patient_id, selected_files[0])

    snapped = max(0, min(idx, base_vol.shape[axis] - 1))
    selected_files = selected_files or []
    overlay_vols = [load_patient_overlay(patient_id, f) for f in selected_files]

    per_overlay_settings = [
        settings.get(f, {"opacity": DEFAULT_OPACITY, "threshold": DEFAULT_THRESHOLD})
        for f in selected_files
    ]
    grad_indices = [_fname_to_grad.get(f, 0) for f in selected_files]

    if mode == "single":
        rgb = blend_single_slice(
            base_vol=base_vol,
            overlay_vols=overlay_vols,
            axis=axis,
            idx=snapped,
            per_overlay_settings=per_overlay_settings,
            grad_indices=grad_indices,
            gradient_settings=gradient_settings,
            selected_files=selected_files,
        )
        fig = px.imshow(rgb)
        fig.update_layout(
            template="plotly_white",
            margin=dict(l=10, r=10, t=50, b=10),
            title=f"{AXIS_LABELS[axis]} slice {snapped}",
            coloraxis_showscale=False,
        )
        fig.update_xaxes(showticklabels=False)
        fig.update_yaxes(showticklabels=False)
        return fig

    mosaic, indices = make_mosaic(base_vol, axis=axis, start=snapped, step=1, n_slices=mosaic_n)
    fig = px.imshow(normalize_to_unit(mosaic), color_continuous_scale="gray")
    fig.update_layout(
        template="plotly_white",
        margin=dict(l=10, r=10, t=50, b=10),
        title=f"Mosaic | {AXIS_LABELS[axis]} | start={snapped} | n={len(indices)}",
        coloraxis_showscale=False,
    )
    fig.update_xaxes(showticklabels=False)
    fig.update_yaxes(showticklabels=False)
    return fig
 

# =========================
# 6) RUN
# =========================
if __name__ == "__main__":
    app.run(debug=False, host="0.0.0.0", port=8051, use_reloader=False)


[2026-03-30 19:22:46,566] ERROR in app: Exception on /_dash-update-component [POST]
Traceback (most recent call last):
  File "/Users/forrestlin/Library/Python/3.11/lib/python/site-packages/flask/app.py", line 1511, in wsgi_app
    response = self.full_dispatch_request()
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/forrestlin/Library/Python/3.11/lib/python/site-packages/flask/app.py", line 919, in full_dispatch_request
    rv = self.handle_user_exception(e)
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/forrestlin/Library/Python/3.11/lib/python/site-packages/flask/app.py", line 917, in full_dispatch_request
    rv = self.dispatch_request()
         ^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/forrestlin/Library/Python/3.11/lib/python/site-packages/flask/app.py", line 902, in dispatch_request
    return self.ensure_sync(self.view_functions[rule.endpoint])(**view_args)  # type: ignore[no-any-return]
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^